In [1]:
import pandas as pd
import unicodedata
import re
import logging
from pathlib import Path
import os
import sys
from os.path import join
import numpy as np
# Import all dirs
parent_dir = Path(os.getcwd()).parents[0]
sys.path.append(str(parent_dir))
from src.paths import all_dirs
dirs = all_dirs()

# ---------- logging setup ----------
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")

# ---------- manual canton mapping ----------
canton_by_central = {
    "Huascachaca":        "Saraguro",            # Loja, cantón Saraguro 
    "Sarapullo":          "Mejia",               # parroquia Manuel Cornejo Astorga (Tandapi), cantón Mejía, Pichincha 
    "Sabanilla":          "Zamora",              # central Sabanilla en cantón Zamora, Zamora Chinchipe 
    "Rio_Verde_Chico":    "Banos_de_Agua_Santa", # parroquia Ulba, cantón Baños de Agua Santa, Tungurahua 
    "Chalpi":             "Quijos",              # proyecto Chalpi en Quijos 
    "San_Jose_de_Tambo":  "Chillanes",           # central San José del Tambo, cantón Chillanes, Bolívar 
    "San_Jose_de_Minas":  "Quito",               # parroquia San José de Minas, Distrito Metropolitano de Quito (cantón Quito) 
    "Ulba":               "Banos_de_Agua_Santa", # Central Hidroeléctrica Ulba, Baños de Agua Santa, Tungurahua 
    "Brineforcorp":       "San_Vicente",         # planta FV Brineforcorp en sitio Briceño, cantón San Vicente, Manabí
    "El_Laurel":          "Mira",                # Central El Laurel, Carchi, cantón Mira 
}

# ---------- ASCII sanitizer for all strings ----------

def to_ascii_token(s: str):
    """
    Make a string ASCII-safe:
    - remove accents
    - replace whitespace with underscore
    - remove quotes and control chars
    - allow only [A-Za-z0-9_.-], everything else -> '_'
    """
    if pd.isna(s):
        return s
    s = str(s)

    # Normalize and strip accents
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    # Remove control chars and quotes (turn into spaces first)
    for ch in ['"', "'", "\r", "\n", "\t"]:
        s = s.replace(ch, " ")

    # Trim and collapse whitespace to single underscores
    s = s.strip()
    s = re.sub(r"\s+", "_", s)

    # Keep only safe chars
    allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_.-")
    s = "".join(ch if ch in allowed else "_" for ch in s)

    # Collapse multiple underscores and trim edges
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")

    return s


def map_sistema(s):
    """Map Sistema to 'SNI' or 'Non_Incorporated'."""
    if isinstance(s, str):
        s_clean = s.strip().upper()
        if "S.N.I" in s_clean or s_clean == "SNI":
            return "SNI"
    return "Non_Incorporated"


def make_location(province, canton):
    """Location before ASCII sanitization, later we sanitize."""
    if pd.isna(province) or pd.isna(canton):
        return pd.NA
    return f"{province}-{canton}"


def ensure_numeric_power(df, pn_col="Potencia Nominal (MW)", pe_col="Potencia Efectiva (MW)"):
    """Make sure PN/PE are numeric (remove thousands apostrophes etc.)."""
    for col in [pn_col, pe_col]:
        if col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                s = df[col].astype(str).str.replace("'", "", regex=False)
                df[col] = pd.to_numeric(s, errors="coerce")
    return df


# ---------- Fill Canton using hard-coded dict ----------

def fill_canton_by_dict(df_target,
                        mapping,
                        source_col="Canton_source"):
    """
    For rows in df_target with missing 'Cantón':
    - compute an ASCII-safe key from 'Central'
    - if key in mapping, set Cantón from dict and mark source_col='manual_dict'
    """
    mask_missing = df_target["Cantón"].isna()
    for idx, row in df_target[mask_missing].iterrows():
        central_key = to_ascii_token(row["Central"])
        if central_key in mapping:
            df_target.at[idx, "Cantón"] = mapping[central_key]
            df_target.at[idx, source_col] = "manual_dict"
    return df_target


# ---------- Auto-fill Canton via power similarity (using 2022 PN/PE vs 2017) ----------

def fill_canton_by_power(df_target, df_ref,
                         pn_col="Potencia Nominal (MW)",
                         pe_col="Potencia Efectiva (MW)",
                         tol=0.05,
                         source_col="Canton_source"):
    """
    For rows in df_target (2022) with missing 'Cantón':
    - use df_ref (2017) with same Provincia
    - match on PN & PE within relative tolerance
    - if found, copy Cantón and mark Canton_source = 'proxy_power'
    """
    # Only consider reference rows that have a Canton and numeric powers
    ref = df_ref[df_ref["Cantón"].notna()].copy()
    ref = ensure_numeric_power(ref, pn_col=pn_col, pe_col=pe_col)

    for idx, row in df_target[df_target["Cantón"].isna()].iterrows():
        prov = row["Provincia"]
        pn = row.get(pn_col)
        pe = row.get(pe_col)

        # Skip if PN/PE are not numeric in 2022 row
        if pd.isna(pn) or pd.isna(pe):
            continue

        # Same province in reference
        cand = ref[ref["Provincia"] == prov].copy()
        if cand.empty:
            continue

        cand = cand[cand[pn_col].notna() & cand[pe_col].notna()]
        if cand.empty:
            continue

        # Relative differences
        rel_pn = (cand[pn_col] - pn).abs() / (pn if pn != 0 else 1.0)
        rel_pe = (cand[pe_col] - pe).abs() / (pe if pe != 0 else 1.0)

        mask = (rel_pn <= tol) & (rel_pe <= tol)
        if not mask.any():
            continue

        score = rel_pn + rel_pe
        best_idx = score[mask].idxmin()

        best_canton = cand.at[best_idx, "Cantón"]
        df_target.at[idx, "Cantón"] = best_canton
        df_target.at[idx, source_col] = "proxy_power"

    return df_target


# ---------- MAIN PIPELINE ----------

file_2022 = join(dirs["data/raw/generation"], "generation_existing_2022.xlsx")
file_2017 = join(dirs["data/raw/generation"] , "generation_existing_2017.xlsx")
file_cantons = join(dirs["data/raw/generation"], "Cantons.xlsx")

df22 = pd.read_excel(file_2022)
df17 = pd.read_excel(file_2017)

# Clean raw column names (strip spaces and newlines)
df22.columns = df22.columns.str.strip().str.replace("\n", " ", regex=False)
df17.columns = df17.columns.str.strip().str.replace("\n", " ", regex=False)

# Align numeric column names if needed
rename_22 = {
    "Potencia Nominal MW": "Potencia Nominal (MW)",
    "Potencia Efectiva MW": "Potencia Efectiva (MW)",
}
df22 = df22.rename(columns={k: v for k, v in rename_22.items() if k in df22.columns})

rename_17 = {
    "Potencia Nominal MW": "Potencia Nominal (MW)",
    "Potencia Efectiva MW": "Potencia Efectiva (MW)",
}
df17 = df17.rename(columns={k: v for k, v in rename_17.items() if k in df17.columns})

# Ensure power columns are numeric in both tables
df22 = ensure_numeric_power(df22)
df17 = ensure_numeric_power(df17)

# Keys for matching (Central + Provincia)
for df in (df22, df17):
    df["Central_key"] = df["Central"].astype(str).str.strip().str.casefold()
    df["Provincia_key"] = df["Provincia"].astype(str).str.strip().str.casefold()

# Merge Cantón from 2017 when Central+Provincia match
df = df22.merge(
    df17[["Central_key", "Provincia_key", "Cantón"]],
    on=["Central_key", "Provincia_key"],
    how="left",
    suffixes=("", "_2017"),
)

df = df.drop(columns=["Central_key", "Provincia_key"])

# Map Sistema
df["Sistema"] = df["Sistema"].apply(map_sistema)

# Canton_source: exact_match or missing (after Central+Provincia merge)
df["Canton_source"] = df["Cantón"].apply(
    lambda x: "exact_match" if pd.notna(x) else "missing"
)

# Pass 1: fill from hard-coded dict (manual)
df = fill_canton_by_dict(
    df_target=df,
    mapping=canton_by_central,
    source_col="Canton_source",
)

# Pass 2: for remaining NaN, infer by power similarity using 2017 as reference
df = fill_canton_by_power(
    df_target=df,
    df_ref=df17,
    pn_col="Potencia Nominal (MW)",
    pe_col="Potencia Efectiva (MW)",
    tol=0.05,
    source_col="Canton_source",
)

# Central_id (ASCII-safe from Central)
df["Central_id"] = df["Central"].apply(to_ascii_token)

# Location (Province-Canton, then later ASCII sanitize)
df["Location"] = df.apply(lambda r: make_location(r["Provincia"], r["Cantón"]), axis=1)

# Technology from Tipo + Subtipo
df["Technology"] = (
    df["Tipo de Central"].astype(str).str.strip()
    + "_"
    + df["Subtipo de Central"].astype(str).str.strip()
)

# Keep only SNI plants
df_out = df[df["Sistema"] == "SNI"].copy()

# ---------- JOIN COORDINATES FROM Cantons.xlsx ----------

df_cant = pd.read_excel(file_cantons, sheet_name= "Canton_Info")
df_cant.columns = df_cant.columns.str.strip().str.replace("\n", " ", regex=False)

# Build merge keys using ASCII-safe tokens (Provincia + Canton)
df_out["Provincia_key"] = df_out["Provincia"].apply(to_ascii_token)
df_out["Canton_key"] = df_out["Cantón"].apply(to_ascii_token)

df_cant["Provincia_key"] = df_cant["Provincia"].apply(to_ascii_token)
df_cant["Canton_key"] = df_cant["Canton"].apply(to_ascii_token)

df_out = df_out.merge(
    df_cant[["Provincia_key", "Canton_key", "Latitud", "Longitud"]],
    on=["Provincia_key", "Canton_key"],
    how="left",
)

df_out = df_out.drop(columns=["Provincia_key", "Canton_key"])

# ---------- FINAL: sanitize EVERYTHING to ASCII tokens ----------

# 1) Sanitize all object columns
obj_cols = df_out.select_dtypes(include="object").columns
for col in obj_cols:
    df_out[col] = df_out[col].apply(to_ascii_token)

# 2) Sanitize column names too
df_out.columns = [to_ascii_token(c) for c in df_out.columns]


# ---------- RIGHT BEFORE SAVING: ensure unique plant names ----------
# Pick the name column you want to enforce uniqueness on
name_col_candidates = ["Central_id"]
name_col = next((c for c in name_col_candidates if c in df_out.columns), None)
if name_col is None:
    raise KeyError(f"No name column found in df_out. Tried: {name_col_candidates}")

# Make sure names are strings (and not NA)
base = df_out[name_col].astype("string")
base = base.fillna("plant")

# For duplicates: keep first as-is, then append 1,2,3...
dup_ix = base.groupby(base).cumcount()  # 0 for first occurrence, 1 for second, ...
df_out[name_col] = np.where(dup_ix == 0, base, base + dup_ix.astype(str))

# Optional sanity check
assert pd.Index(df_out[name_col]).is_unique, f"Still not unique: {name_col}"



# ---------- Save ----------
output_file =join(dirs["data/processed/generation"] , "cleaned_generation_2022.xlsx")
df_out.to_excel(output_file, index=False, sheet_name="Generation")

logging.info("Saved cleaned file to: %s", output_file)


INFO:Saved cleaned file to: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\cleaned_generation_2022.xlsx


In [2]:
import pandas as pd
import unicodedata
import re
import logging

# ---------------------------------------------------------------------
# Logging setup
# ---------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(message)s"
)

# ---------------------------------------------------------------------
# ASCII sanitizer for all strings
# ---------------------------------------------------------------------
def to_ascii_token(s: str):
    """
    Make a string ASCII-safe:
    - remove accents
    - replace whitespace with underscore
    - remove quotes and control chars
    - allow only [A-Za-z0-9_.-], everything else -> '_'
    """
    if pd.isna(s):
        return s
    s = str(s)

    # Normalize and strip accents
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    # Remove control chars and quotes (turn into spaces first)
    for ch in ['"', "'", "\r", "\n", "\t"]:
        s = s.replace(ch, " ")

    # Trim and collapse whitespace to single underscores
    s = s.strip()
    s = re.sub(r"\s+", "_", s)

    # Keep only safe chars
    allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_.-")
    s = "".join(ch if ch in allowed else "_" for ch in s)

    # Collapse multiple underscores and trim edges
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")

    return s

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def map_sistema_future():
    """All future plants assumed to be connected to SNI."""
    return "SNI"


def ensure_numeric_power(df, pn_col="Potencia_Nominal_MW"):
    """Make sure power column is numeric (remove thousands apostrophes etc.)."""
    if pn_col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[pn_col]):
            s = df[pn_col].astype(str).str.replace("'", "", regex=False)
            df[pn_col] = pd.to_numeric(s, errors="coerce")
    return df


def normalize_tipo_key(s):
    """Normalize 'Tipo' to a simple lowercase ASCII key like 'hidroelectrico'."""
    if pd.isna(s):
        return None
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.strip().lower()


# tipo (from source) -> Tipo_de_Central / Subtipo_de_Central
tipo_main_map = {
    "hidroelectrico": "Hidraulica",
    "hidroelectrica": "Hidraulica",
    "eolico":         "Eolica",
    "eolica":         "Eolica",
    "fotovoltaico":   "Solar",
    "fotovoltaica":   "Solar",
    "termoelectrico": "Termica",
    "termico":        "Termica",
}

tipo_sub_map = {
    # Assumptions: future small/medium hydro = Pasada,
    # generic thermal firm blocks = Turbogas
    "hidroelectrico": "Pasada",
    "hidroelectrica": "Pasada",
    "eolico":         "Eolica",
    "eolica":         "Eolica",
    "fotovoltaico":   "Solar",
    "fotovoltaica":   "Solar",
    "termoelectrico": "Turbogas",
    "termico":        "Turbogas",
}


def map_tipo_to_central(df, tipo_col="Tipo"):
    """Fill Tipo_de_Central and Subtipo_de_Central from 'Tipo'."""
    tipo_norm = df[tipo_col].apply(normalize_tipo_key)
    df["Tipo_de_Central"] = tipo_norm.map(tipo_main_map).fillna("Other")
    df["Subtipo_de_Central"] = tipo_norm.map(tipo_sub_map).fillna("Other")
    return df


def make_location(province, canton):
    """Location string before ASCII sanitization."""
    if pd.isna(province) or pd.isna(canton):
        return pd.NA
    return f"{province}-{canton}"


# ---------------------------------------------------------------------
# 1) Read FUTUREPLANTS from generation_future_2022.xlsx
# ---------------------------------------------------------------------
file_future =join(dirs["data/raw/generation"], "generation_future_2022.xlsx")
sheet_future = "FuturePlants"

df_fut = pd.read_excel(file_future, sheet_name=sheet_future)

# Clean column names
df_fut.columns = df_fut.columns.str.strip().str.replace("\n", " ", regex=False)

# Rename to standard internal names
rename_fut = {
    "Ano de entrada en operacion": "Year",
    "Año de entrada en operacion": "Year",
    "Proyecto / Central": "Central",
    "Ennpresa / InsItucion": "Empresa",
    "Empresa / InsItucion": "Empresa",
    "Tipo": "Tipo",
    "Power": "Potencia_Nominal_MW",
    "Energy": "Energy",
    "Provincia": "Provincia",
    "Canton": "Canton",
    "Filter": "Filter",
}
df_fut = df_fut.rename(columns=rename_fut)

# Keep only rows with Filter == "Yes"
if "Filter" in df_fut.columns:
    df_fut = df_fut[
        df_fut["Filter"].astype(str).str.strip().str.lower() == "yes"
    ].copy()

# Ensure power numeric
df_fut = ensure_numeric_power(df_fut, pn_col="Potencia_Nominal_MW")

# Sistema
df_fut["Sistema"] = map_sistema_future()

# Map Tipo -> Tipo_de_Central / Subtipo_de_Central
df_fut = map_tipo_to_central(df_fut, tipo_col="Tipo")

# Effective power (best guess: same as nominal)
df_fut["Potencia_Efectiva_MW"] = df_fut["Potencia_Nominal_MW"]

# Canton_source
df_fut["Canton_source"] = "future_main"



# Year as numeric
df_fut["Year"] = pd.to_numeric(df_fut["Year"], errors="coerce")

# ---------------------------------------------------------------------
# 2) Read UNDEFINED_GENERATION.CSV
# ---------------------------------------------------------------------
file_undef = join(dirs["data/raw/generation"], "generation_future_renewable_2022.csv") 
df_und = pd.read_csv(file_undef)

df_und.columns = df_und.columns.str.strip().str.replace("\n", " ", regex=False)

rename_und = {
    "Ano de entrada en operacion": "Year",
    "Año de entrada en operacion": "Year",
    "Proyecto / Central": "Central",
    "Empresa / Institucion": "Empresa",
    "Empresa / Institución": "Empresa",
    "Tipo": "Tipo",
    "Power": "Potencia_Nominal_MW",
    "Energy": "Energy",
    "Provincia": "Provincia",
    "Canton": "Canton",
}
df_und = df_und.rename(columns=rename_und)

# Ensure power numeric
df_und = ensure_numeric_power(df_und, pn_col="Potencia_Nominal_MW")

# Sistema
df_und["Sistema"] = map_sistema_future()

# Map Tipo -> Tipo_de_Central / Subtipo_de_Central
df_und = map_tipo_to_central(df_und, tipo_col="Tipo")

# Effective power (best guess: same as nominal)
df_und["Potencia_Efectiva_MW"] = df_und["Potencia_Nominal_MW"]

# Canton_source
df_und["Canton_source"] = "future_undefined"



# Year as numeric
df_und["Year"] = pd.to_numeric(df_und["Year"], errors="coerce")

# ---------------------------------------------------------------------
# 3) Harmonize columns & concatenate future datasets
# ---------------------------------------------------------------------
# Ensure both have Empresa, Central, Provincia, Canton, Tipo_de_Central, Subtipo_de_Central, etc.
needed_cols = [
    "Empresa", "Central", "Provincia", "Sistema",
    "Tipo_de_Central", "Subtipo_de_Central",
    "Potencia_Nominal_MW", "Potencia_Efectiva_MW",
    "Canton", "Canton_source", "Year"
]

for col in needed_cols:
    if col not in df_fut.columns:
        df_fut[col] = pd.NA
    if col not in df_und.columns:
        df_und[col] = pd.NA

df_future_all = pd.concat(
    [df_fut[needed_cols + ["Energy"] if "Energy" in df_fut.columns else needed_cols],
     df_und[needed_cols + ["Energy"] if "Energy" in df_und.columns else needed_cols]],
    ignore_index=True
)

# Drop Energy from final (not in existing format)
if "Energy" in df_future_all.columns:
    df_future_all = df_future_all.drop(columns=["Energy"])

# ---------------------------------------------------------------------
# 4) Central_id, Location, Technology
# ---------------------------------------------------------------------
df_future_all["Central_id"] = df_future_all["Central"].apply(to_ascii_token)

df_future_all["Location"] = df_future_all.apply(
    lambda r: make_location(r["Provincia"], r["Canton"]),
    axis=1
)

df_future_all["Technology"] = (
    df_future_all["Tipo_de_Central"].astype(str).str.strip()
    + "_"
    + df_future_all["Subtipo_de_Central"].astype(str).str.strip()
)

# ---------------------------------------------------------------------
# 5) Join coordinates from Cantons.xlsx using Provincia + Canton
# ---------------------------------------------------------------------
file_cantons = join(dirs["data/raw/generation"],"Cantons.xlsx")
df_cant = pd.read_excel(file_cantons)
df_cant.columns = df_cant.columns.str.strip().str.replace("\n", " ", regex=False)

# Build merge keys using ASCII-safe tokens
df_future_all["Provincia_key"] = df_future_all["Provincia"].apply(to_ascii_token)
df_future_all["Canton_key"]   = df_future_all["Canton"].apply(to_ascii_token)

df_cant["Provincia_key"] = df_cant["Provincia"].apply(to_ascii_token)
df_cant["Canton_key"]    = df_cant["Canton"].apply(to_ascii_token)

df_future_all = df_future_all.merge(
    df_cant[["Provincia_key", "Canton_key", "Latitud", "Longitud"]],
    on=["Provincia_key", "Canton_key"],
    how="left",
)

df_future_all = df_future_all.drop(columns=["Provincia_key", "Canton_key"])

# If no coordinates found, put placeholders so you can fix later
df_future_all["Latitud"] = df_future_all["Latitud"].where(
    df_future_all["Latitud"].notna(), "TODO_LAT"
)
df_future_all["Longitud"] = df_future_all["Longitud"].where(
    df_future_all["Longitud"].notna(), "TODO_LON"
)

# ---------------------------------------------------------------------
# 6) FINAL: sanitize EVERYTHING string-like to ASCII tokens
# ---------------------------------------------------------------------
obj_cols = df_future_all.select_dtypes(include="object").columns
for col in obj_cols:
    df_future_all[col] = df_future_all[col].apply(to_ascii_token)

# Also sanitize column names
df_future_all.columns = [to_ascii_token(c) for c in df_future_all.columns]

# ---------------------------------------------------------------------
# 7) Reorder columns to match existing format + Year at the end
# ---------------------------------------------------------------------
column_order = [
    "Empresa",
    "Central",
    "Provincia",
    "Sistema",
    "Tipo_de_Central",
    "Subtipo_de_Central",
    "Potencia_Nominal_MW",
    "Potencia_Efectiva_MW",
    "Canton",
    "Canton_source",
    "Central_id",
    "Location",
    "Technology",
    "Latitud",
    "Longitud",
    "Year",
]

# Ensure all exist
for col in column_order:
    if col not in df_future_all.columns:
        df_future_all[col] = pd.NA

df_future_all = df_future_all[column_order]

# ---------------------------------------------------------------------
# 8) Save to Excel
# ---------------------------------------------------------------------
output_file = join(dirs["data/processed/generation"], "cleaned_generation_future_2022.xlsx")

df_future_all.to_excel(output_file, index=False, sheet_name="GenerationFuture")


logging.info("Saved future generation file to: %s", output_file)
non_located = df_future_all[df_future_all["Latitud"]=="TODO_LAT"]["Location"].unique()
assert len(non_located) == 0, f"Some future plants have missing coordinates: {non_located}"


INFO:Saved future generation file to: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\cleaned_generation_future_2022.xlsx
